In [5]:
# Implementation of a short straddle strategy selling ATM options expiry T = 3 months, no buy backs
# Daily PnL is calculated using capital gains of option premium 

'''
Sell ATM options on days when the portfolio doesn't already have open positions.
Rebalance the portfolio daily based on delta, either buying or short selling the underlying stock.
Calculate daily returns, taking into account the change in option premiums and the profit or loss from rebalancing the underlying stock.
Handle option expiration by checking if the options in the portfolio are expiring on the current day and calculating returns based on whether the options are exercised.
'''

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math as math
from pandas import Timestamp
from equity import find_atm, reverse_option, get_hedge_count


In [18]:
# Load Data
AAPL_options_csv = 'AAPL_Options_22-23.csv'
AAPL_underlying_csv = 'AAPL_Underlying_22-23.csv'
AAPL_options = pd.read_csv(AAPL_options_csv)
AAPL_underlying = pd.read_csv(AAPL_underlying_csv)
find_atm(AAPL_options_csv, AAPL_underlying_csv, pd.Timestamp(2022, 2, 28), 10, 90, 0.5)


[('AAPL 220520C155000', 'AAPL 220520P155000'),
 ('AAPL 220520P155000', 'AAPL 220520C155000'),
 ('AAPL 220520C160000', 'AAPL 220520P160000'),
 ('AAPL 220520P160000', 'AAPL 220520C160000'),
 ('AAPL 220520C165000', 'AAPL 220520P165000'),
 ('AAPL 220520P165000', 'AAPL 220520C165000'),
 ('AAPL 220520P130000', 'AAPL 220520C130000'),
 ('AAPL 220520C130000', 'AAPL 220520P130000'),
 ('AAPL 220520C185000', 'AAPL 220520P185000'),
 ('AAPL 220520P185000', 'AAPL 220520C185000')]

In [29]:

def backtest_short_straddle_with_premium_change(options_df, underlying_df, start_date, end_date):
    portfolio = {'options': [], 'underlying': 0, 'cash': 0}
    daily_returns = []

    for current_date in pd.date_range(start_date, end_date):
        daily_unrealized_pnl = 0  # Initialize daily unrealized profit/loss from options
        
        # Process options expiring today
        options_to_remove = []
        for option in portfolio['options']:
            option_data = options_df.loc[options_df['symbol'] == option['symbol']].iloc[0]
            if option_data['exdate'] == current_date:
                # Calculate and realize P&L from option expiration
                spot_price = underlying_df.loc[underlying_df['date'] == current_date, 'PRC'].item()
                pnl = max(option_data['strike'] - spot_price, 0) if 'P' in option_data['symbol'] else max(spot_price - option_data['strike'], 0)
                pnl *= option_data['contract_size']
                portfolio['cash'] += pnl - option['premium_received']
                options_to_remove.append(option)
            else:
                # Calculate unrealized P&L for non-expiring options
                current_premium = (option_data['best_bid'] + option_data['best_offer']) / 2
                premium_change = option['premium_received'] - current_premium * option_data['contract_size']
                daily_unrealized_pnl += premium_change

        # Remove expired options
        for option in options_to_remove:
            portfolio['options'].remove(option)

        # Sell new ATM options if needed
        if not portfolio['options']:
            atm_options = find_atm(AAPL_options_csv, AAPL_underlying_csv, current_date, 10, 90, 0.5)
            for call_id, put_id in atm_options:
                call_data = options_df.loc[options_df['symbol'] == call_id].iloc[0]
                put_data = options_df.loc[options_df['symbol'] == put_id].iloc[0]
                call_premium = (call_data['best_bid'] + call_data['best_offer']) / 2 * call_data['contract_size']
                put_premium = (put_data['best_bid'] + put_data['best_offer']) / 2 * put_data['contract_size']
                portfolio['options'].append({'symbol': call_id, 'premium_received': call_premium})
                portfolio['options'].append({'symbol': put_id, 'premium_received': put_premium})
                portfolio['cash'] += call_premium + put_premium

        # Daily rebalance based on delta
        portfolio_option_symbols = [option['symbol'] for option in portfolio['options']]
            
        hedge_count = get_hedge_count(options_df, portfolio_option_symbols, current_date)
        rebalance_cost = (hedge_count - portfolio['underlying']) * underlying_df.loc[underlying_df['date'] == current_date, 'PRC'].item()
        portfolio['underlying'] = hedge_count
        portfolio['cash'] -= rebalance_cost

        # Add daily unrealized P&L from options to daily returns
        daily_returns.append(daily_unrealized_pnl - rebalance_cost)

    return pd.Series(data=daily_returns, index=pd.date_range(start_date, end_date))

# Assuming find_atm, get_hedge_count functions are defined, and data frames are populated
# Initialize your options_df, underlying_df, start_date, and end_date accordingly

# Execute the backtest
daily_returns = backtest_short_straddle_with_premium_change(AAPL_options, AAPL_underlying, AAPL_options['date'].min(), AAPL_options['date'].max())

['AAPL 220520C155000', 'AAPL 220520P155000', 'AAPL 220520P155000', 'AAPL 220520C155000', 'AAPL 220520C160000', 'AAPL 220520P160000', 'AAPL 220520P160000', 'AAPL 220520C160000', 'AAPL 220520C165000', 'AAPL 220520P165000', 'AAPL 220520P165000', 'AAPL 220520C165000', 'AAPL 220520P130000', 'AAPL 220520C130000', 'AAPL 220520C130000', 'AAPL 220520P130000', 'AAPL 220520C185000', 'AAPL 220520P185000', 'AAPL 220520P185000', 'AAPL 220520C185000']


/Users/nicolas/Documents/equitymarkets-project/equity.py:209: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print("here")


KeyError: 0